In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path

CANDIDATES = [
    Path("../data/processed/price_review_task4"),
    Path("data/processed/price_review_task4"),
    Path("."),
    Path(".."),
]

DATA_DIR = next(
    (
        p / "price_review_task4"
        if (p / "price_review_task4").is_dir()
        else p
        for p in CANDIDATES
        if (
            (p / "price_review_task4").is_dir()
            or (
                p.name == "price_review_task4"
                and p.is_dir()
            )
        )
    ),
    None
)



print("DATA_DIR:", DATA_DIR.resolve())
print("Files:", [f.name for f in DATA_DIR.iterdir()])

DATA_DIR: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\price_review_task4
Files: ['train_scored_4_1.csv', 'val_scored_4_1.csv']


In [3]:
train_s = pd.read_csv(DATA_DIR / "train_scored_4_1.csv")
val_s = pd.read_csv(DATA_DIR / "val_scored_4_1.csv")

print("train:", train_s.shape, "| val:", val_s.shape)

train: (5165, 64) | val: (1252, 64)


In [ ]:
rows = []
for name, d in [("train", train_s), ("val", val_s)]:
    unk = d["model_key"].eq("unknown")
    has_ref = d["ref_n"].notna()

    rows.append({
        "split": name,
        "ads": len(d),
        "unknown_model": int(unk.sum()),
        "unknown_%": round(unk.mean() * 100, 1),
        "known_but_thin_reference": int((~unk & ~has_ref).sum()),
        "with_reference": int(has_ref.sum()),
        "with_reference_%": round(has_ref.mean() * 100, 1),
        "flagged": int(d["price_flag"].sum()),
        "flagged_%": round(d["price_flag"].mean() * 100, 2),
    })

summary = pd.DataFrame(rows)
display(summary)

,split,ads,unknown_model,unknown_%,known_but_thin_reference,with_reference,with_reference_%,flagged,flagged_%
0,train,5165,465,9.0,733,3967,76.8,191,3.70
1,val,1252,122,9.7,280,850,67.9,49,3.91


In [10]:
print("DATA_DIR:", DATA_DIR.resolve())

for f in DATA_DIR.glob("*.csv"):
    print(f.name)

DATA_DIR: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\price_review_task4
train_scored_4_1.csv
val_scored_4_1.csv


الإعلانات اللي عندها مرجع سعري على مستوى الموديل، واقدر اقارن سعره بالسعر المرجعي

In [13]:

# 1. Select TRAIN ads whose price reference is based on model_key
eligible_train_cases = train_s[(train_s["ref_level"] == "model_key") & train_s["price_ratio"].notna()].copy()

# Keep only valid numeric ratios
eligible_train_cases["price_ratio"] = pd.to_numeric( eligible_train_cases["price_ratio"], errors="coerce")

eligible_train_cases = eligible_train_cases[ eligible_train_cases["price_ratio"].notna() & np.isfinite(eligible_train_cases["price_ratio"]) ].sort_values("price_ratio")



# 2. Make sure we have enough cases
if len(eligible_train_cases) < 100:
    raise ValueError(
        f"Only {len(eligible_train_cases)} eligible cases found; need at least 100."
    )

# 3. Select the 50 lowest and 50 highest ratios
lowest = eligible_train_cases.head(50).copy()
lowest["side"] = "lowest"

highest = eligible_train_cases.tail(50).iloc[::-1].copy()
highest["side"] = "highest"

review = pd.concat(
    [lowest, highest],
    ignore_index=True
)


In [16]:
cols = [
    "ad_id", "ad_url", "side", "title", "brand", "model_key",
    "price", "ref_median_price", "price_ratio", "mad_z",
    "price_review_reason", "price_context"
]

review_sheet = review[cols].round(2).copy()

for col in ["decision", "fixed_price", "fixed_model_key", "error_type", "notes"]:
    review_sheet[col] = ""

print(f"Eligible TRAIN cases: {len(eligible_train_cases)}")
print(f"Exported cases: {len(review_sheet)}")

for side in ["lowest", "highest"]:
    ratios = review.loc[review["side"] == side, "price_ratio"]
    print(f"{side.title()} ratio range: {ratios.min():.3f} - {ratios.max():.3f}")

display(review_sheet.head(8))

Eligible TRAIN cases: 3967
Exported cases: 100
Lowest ratio range: 0.012 - 0.135
Highest ratio range: 1.719 - 9.286


,ad_id,ad_url,side,title,brand,model_key,price,ref_median_price,price_ratio,mad_z,price_review_reason,price_context,decision,fixed_price,fixed_model_key,error_type,notes
0,208989958,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1000.0,83000.0,0.01,-29.81,price_placeholder;ratio_low;mad_low,,,,,,
1,208954191,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1000.0,83000.0,0.01,-29.81,price_placeholder;ratio_low;mad_low,,,,,,
2,208983975,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو,apple,apple iphone 17 pro,1000.0,74000.0,0.01,-20.95,price_placeholder;ratio_low;mad_low,,,,,,
3,209009851,/ad/%D8%B3%D8%A7%D9%85%D8%B3%D9%88%D9%86%D8%AC...,lowest,سامسونج جلاكسي اس 26 الترا,samsung,samsung s26 ultra,1000.0,67000.0,0.01,-28.36,price_placeholder;ratio_low;mad_low,,,,,,
4,208942594,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1500.0,83000.0,0.02,-27.07,ratio_low;mad_low,,,,,,
5,208984516,/ad/%D8%B9%D8%B1%D8%B1%D8%B1%D8%B6-%D8%AE%D8%B...,lowest,عرررض خطيررر ع الايفونن 17 برماكس *IPHoooONE ...,apple,apple iphone 17 pro max,2000.0,83000.0,0.02,-25.13,ratio_low;mad_low,,,,,,
6,208986825,/ad/%D8%A7%D9%8A%D9%88%D8%A7%D8%A7%D8%A7%D8%A7...,lowest,ايواااا الجديد شديد ايفون١٧بروووووماكس *iPhon...,apple,apple iphone 17 pro max,2500.0,83000.0,0.03,-23.62,ratio_low;mad_low,,,,,,
7,209005186,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 16 برو ماكس,apple,apple iphone 16 pro max,2000.0,63000.0,0.03,-23.27,ratio_low;mad_low,,,,,,


In [17]:
review_sheet.columns.tolist()

['ad_id',
 'ad_url',
 'side',
 'title',
 'brand',
 'model_key',
 'price',
 'ref_median_price',
 'price_ratio',
 'mad_z',
 'price_review_reason',
 'price_context',
 'decision',
 'fixed_price',
 'fixed_model_key',
 'error_type',
 'notes']